In [1]:
import pandas as pd
import numpy as np

import os
import sys
sys.path.append(os.path.abspath('./src'))

from dota_data_manager import DotaDataManager
from db_functions import DotaDB

import logging
import basic_logger
basic_logger.setup_logger()
db = DotaDB()
dota_data = DotaDataManager(db)

INFO: HTTP Request: GET https://api.opendota.com/api/leagues "HTTP/1.1 200 OK"
INFO: HTTP Request: GET https://api.opendota.com/api/leagues "HTTP/1.1 200 OK"


In [ ]:
dl_leagues = dota_data.dreamleague_leagues
for league in dl_leagues:
    if 'DreamLeague Season 28' == league['name']:
        league_id = league['id']
        break

In [16]:
live_matches = db.query_opendota('live')
for match in live_matches:
    if match['league_id'] == league_id:
        match_id = match['match_id']
        break

INFO: HTTP Request: GET https://api.opendota.com/api/live "HTTP/1.1 200 OK"
INFO: HTTP Request: GET https://api.opendota.com/api/live "HTTP/1.1 200 OK"


In [ ]:
query = '''
    query($id: Long!) {
        live {
            match(id: $id) {
                
            }
        }
    }
'''

In [ ]:
main_league_ids = dota_data.main_leagues
query = '''
    query($leagueIds: [Int]) {
        leagues(request: {leagueIds: $leagueIds}){
            id
            hasLiveMatches
        }
    }
'''
results = db.query_stratz(query, variables={'leagueIds': main_league_ids})['data']['leagues']
live_leagues = []
for league in results:
    if league['hasLiveMatches']:
        live_leagues.append(league['id'])

INFO: HTTP Request: POST https://api.stratz.com/graphql "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.stratz.com/graphql "HTTP/1.1 200 OK"


In [ ]:

query = '''
    query($leagueId: Int) {
        live {
            matches(request: {leagueId: $leagueId}) {
                matchId
            }
        }
    }
'''
results = db.query_stratz(query, variables={'leagueId': league_id})
results


INFO: HTTP Request: POST https://api.stratz.com/graphql "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.stratz.com/graphql "HTTP/1.1 200 OK"


{'data': {'live': {'matches': []}}}

In [4]:
results

{'data': {'live': {'matches': []}}}

In [ ]:
query = '''
    query($leagueId: Int) {
        leagues(request: {leagueId: $leagueId}){
            id
            displayName
            hasLiveMatches
        }
    }
'''
results = db.query_stratz(query, variables={'leagueId': league_id})['data']['leagues']
results